In [ ]:
from datetime import datetime
from pathlib import Path
import csv
import cv2
import face_recognition
# isme is file ka path dal dena
KNOWN_FACES_DIR = Path(r"add your knoun_face dir path")
# or isme filepath\file name 
CSV_FILE = Path(r"add path where csvfile was create in end add fime name after '\'")
TOLERANCE = 0.50
CAMERA_INDEX = 0

def load_known_faces():
    encodings = []
    names = []

    for image_path in KNOWN_FACES_DIR.iterdir():
        if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        image = face_recognition.load_image_file(image_path)
        found_encodings = face_recognition.face_encodings(image)

        if not found_encodings:
            print(f"No face found in {image_path.name}; skipping.")
            continue

        encodings.append(found_encodings[0])
        names.append(image_path.stem)

    if not encodings:
        raise RuntimeError(f"No usable face images found in {KNOWN_FACES_DIR.resolve()}")

    return encodings, names

def record_once_per_day(person_id):
    today = datetime.now().date().isoformat()
    already_recorded = set()
    if CSV_FILE.exists():
        with CSV_FILE.open("r", newline="", encoding="utf-8") as file:
            for row in csv.DictReader(file):
                if row.get("date") == today:
                    already_recorded.add(row.get("person_id"))
    if person_id in already_recorded:
        return False
    file_exists = CSV_FILE.exists()
    with CSV_FILE.open("a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["person_id", "date", "time"])
        if not file_exists:
            writer.writeheader()

        writer.writerow({
            "person_id": person_id,
            "date": today,
            "time": datetime.now().strftime("%H:%M:%S"),
        })

    return True


def main():
    known_encodings, known_ids = load_known_faces()
    camera = cv2.VideoCapture(CAMERA_INDEX)

    if not camera.isOpened():
        raise RuntimeError("Could not open the camera.")

    print("Press q in the camera window to stop.")
    try:
        while True:
            ok, frame = camera.read()
            if not ok:
                print("Could not read from the camera.")
                break

            small = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
            rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)

            locations = face_recognition.face_locations(rgb)
            encodings = face_recognition.face_encodings(rgb, locations)

            for (top, right, bottom, left), encoding in zip(locations, encodings):
                distances = face_recognition.face_distance(known_encodings, encoding)
                best_index = distances.argmin()

                if distances[best_index] <= TOLERANCE:
                    person_id = known_ids[best_index]
                    if record_once_per_day(person_id):
                        label = f"{person_id} - marked present"
                        print(label)
                    else:
                        label = f"{person_id} - already marked"
                else:
                    label = "Unknown"
                top, right, bottom, left = [v * 4 for v in (top, right, bottom, left)]
                cv2.rectangle(frame, (left, top), (right, bottom), (0, 200, 0), 2)
                cv2.putText(
                    frame, label, (left, max(20, top - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 2
                )
            cv2.imshow("Attendance (press q to quit)", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        camera.release()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

C:\Users\keshav prasad sinha\anaconda3\envs\sihenv\Lib\site-packages\face_recognition_models\__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Press q in the camera window to stop.
shail - marked present
neelam - marked present
